# Experiment 4 — Chain-of-Thought (CoT), Qwen

**Model:** `qwen/qwen3.8-27b` via Groq API  
**Benchmark:** Same fixed 250-question obfuscated benchmark used in Experiments 1–3.

| Task | Variant | Questions |
|---|---|---:|
| FOL | obfuscation | 50 |
| Blood Relation | obfuscation_l1 | 50 |
| Blood Relation | obfuscation_l2 | 50 |
| Direction Sense | obfuscation | 50 |
| Number Series | obfuscation | 50 |
| **Total** | | **250** |

**Method:** Chain-of-Thought prompting asks the model to reason carefully through facts and constraints, but return only the final answer.

Run a 5-question test first. Only then switch `TEST_MODE = False` for the full benchmark.

In [ ]:

!pip -q install pandas openai tqdm

In [ ]:

import os, re, time, random
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from IPython.display import display

# True = 5-question test; False = full 250-question inference
TEST_MODE = True
TEST_SAMPLES_PER_VARIANT = 1

MODEL_NAME = "qwen/qwen3.8-27b"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
TEMPERATURE = 0
MAX_TOKENS = 500
CHECKPOINT_EVERY_SUCCESSFUL = 10
REQUEST_DELAY_SECONDS = 1.5
MAX_RETRIES = 5

print("Model:", MODEL_NAME)
print("TEST_MODE:", TEST_MODE)

In [ ]:

GROQ_API_KEY = None
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass("Enter your GROQ_API_KEY: ")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is required.")

client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BASE_URL)
print("Groq client initialized.")

In [ ]:

from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file was uploaded.")
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise ValueError("Please upload the 250-question benchmark CSV.")
csv_name = csv_names[0]
df = pd.read_csv(csv_name)
print("Loaded:", csv_name, "| Rows:", len(df))
print("Columns:", list(df.columns))
display(df.head())

In [ ]:

required = ["dataset", "variant", "prompt", "ground_truth", "response_id"]
missing = [x for x in required if x not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
expected = {
    ("FOL", "obfuscation"): 50,
    ("Blood_Relation", "obfuscation_l1"): 50,
    ("Blood_Relation", "obfuscation_l2"): 50,
    ("Direction_Sense", "obfuscation"): 50,
    ("Number_Series", "obfuscation"): 50,
}
distribution = df.groupby(["dataset", "variant"]).size().reset_index(name="count")
display(distribution)
for (ds, var), n in expected.items():
    actual = int(((df["dataset"].astype(str) == ds) & (df["variant"].astype(str) == var)).sum())
    print(f"{ds} / {var}: {actual}/{n}")
    if actual != n:
        raise ValueError(f"Expected {n} rows for {ds}/{var}; found {actual}")
if len(df) != 250:
    raise ValueError(f"Expected 250 rows, found {len(df)}")
if df["response_id"].astype(str).duplicated().any():
    raise ValueError("response_id values are not unique")
if df["ground_truth"].fillna("").astype(str).str.strip().eq("").any():
    raise ValueError("At least one ground truth is missing")
print("✅ Fixed 250-question benchmark validated.")

In [ ]:

COT_SYSTEM = r"""
You are solving an obfuscated logical reasoning problem.

Use careful Chain-of-Thought reasoning internally:
1. Read the entire problem, including all conditions and wording.
2. Identify relevant entities, facts, relationships, quantities, rules, and constraints.
3. Work through the logical or numerical implications step by step.
4. Check the proposed answer against the original problem and all relevant constraints.
5. Correct any contradiction or calculation issue before answering.

Do not assume facts that are not stated. Preserve the meaning of the original question.
Return ONLY the final answer requested by the question. Do not output reasoning, explanations, labels, or extra text.
"""
print("CoT prompt ready.")

In [ ]:

def normalize_answer(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value).strip()
    prefixes = ["Final Answer:", "Final answer:", "Answer:", "answer:", "The answer is:", "the answer is:"]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
            break
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text.strip(" \t\n\r\"'.,;:!?")

def run_model(prompt):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": COT_SYSTEM}, {"role": "user", "content": str(prompt)}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    content = response.choices[0].message.content
    return "" if content is None else str(content).strip()

def is_transient_error(exc):
    msg = repr(exc).lower()
    return any(term in msg for term in ["429", "rate limit", "too many requests", "timeout", "temporarily", "502", "503", "504", "service unavailable"])

print("Inference helpers ready.")

In [ ]:

variant_order = [
    ("FOL", "obfuscation"),
    ("Blood_Relation", "obfuscation_l1"),
    ("Blood_Relation", "obfuscation_l2"),
    ("Direction_Sense", "obfuscation"),
    ("Number_Series", "obfuscation"),
]
if TEST_MODE:
    parts = []
    for ds, var in variant_order:
        part = df[(df["dataset"].astype(str) == ds) & (df["variant"].astype(str) == var)].head(TEST_SAMPLES_PER_VARIANT)
        if len(part) != TEST_SAMPLES_PER_VARIANT:
            raise ValueError(f"Could not select test row for {ds}/{var}")
        parts.append(part)
    run_df = pd.concat(parts, ignore_index=True)
    print("TEST MODE — rows to process:", len(run_df))
else:
    run_df = df.copy().reset_index(drop=True)
    print("FULL MODE — rows to process:", len(run_df))
    if len(run_df) != 250:
        raise ValueError(f"Expected 250 rows, found {len(run_df)}")
display(run_df[["dataset", "variant", "row_number", "response_id", "ground_truth"]])

In [ ]:

checkpoint_file = None if TEST_MODE else "experiment_04_chain_of_thought_Qwen_250_checkpoint.csv"
results = []
if not TEST_MODE and os.path.exists(checkpoint_file):
    previous = pd.read_csv(checkpoint_file)
    previous["prediction"] = previous["prediction"].fillna("").astype(str).str.strip()
    completed_previous = previous[previous["prediction"] != ""].copy()
    results = completed_previous.to_dict("records")
    processed_ids = set(completed_previous["response_id"].astype(str).tolist())
    print(f"Resuming from checkpoint: {len(completed_previous)} successful predictions")
else:
    processed_ids = set()
    if not TEST_MODE:
        print("No usable checkpoint found; starting full run")

for _, row in tqdm(run_df.iterrows(), total=len(run_df), desc="Qwen Chain-of-Thought"):
    rid = str(row["response_id"])
    if not TEST_MODE and rid in processed_ids:
        continue
    prediction, error = "", ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            prediction = run_model(row["prompt"])
            if prediction.strip():
                error = ""
                break
            error = "EMPTY_MODEL_RESPONSE"
        except Exception as exc:
            error = repr(exc)
            if attempt < MAX_RETRIES:
                wait = min(60, 5 * (2 ** (attempt - 1))) + random.uniform(0, 2)
                if is_transient_error(exc):
                    print(f"\nRate-limit/transient error for {rid}; waiting {wait:.1f}s")
                    time.sleep(wait)
                else:
                    print(f"\nAPI error for {rid}, attempt {attempt}: {exc}")
                    time.sleep(2)
            else:
                print(f"\nFailed after {MAX_RETRIES} attempts: {rid}")
    gt = normalize_answer(row["ground_truth"])
    pred = normalize_answer(prediction)
    results.append({
        "dataset": row["dataset"], "variant": row["variant"],
        "setting": row.get("setting", ""), "row_number": row.get("row_number", ""),
        "response_id": rid, "ground_truth": gt, "prediction": pred,
        "correct": bool(gt and pred and gt == pred), "error": error if not pred else "",
    })
    if not TEST_MODE:
        temp = pd.DataFrame(results)
        success_n = int(temp["prediction"].fillna("").astype(str).str.strip().ne("").sum())
        if success_n > 0 and success_n % CHECKPOINT_EVERY_SUCCESSFUL == 0:
  Agree and access repository,          temp.to_csv(checkpoint_file, index=False)
            print(f"\nCheckpoint saved: {success_n}/250 successful predictions")
    time.sleep(REQUEST_DELAY_SECONDS)

results_df = pd.DataFrame(results)
print("\nInference pass complete")
print("Rows in results:", len(results_df))
print("Predictions:", int(results_df["prediction"].fillna("").astype(str).str.strip().ne("").sum()))
print("Missing predictions:", int(results_df["prediction"].fillna("").astype(str).str.strip().eq("").sum()))
print("Errors:", int(results_df["error"].fillna("").astype(str).str.strip().ne("").sum()))

In [ ]:

display(results_df[["dataset", "variant", "row_number", "ground_truth", "prediction", "correct", "error"]])

In [ ]:

results_df["prediction"] = results_df["prediction"].fillna("").astype(str).str.strip()
results_df["error"] = results_df["error"].fillna("").astype(str).str.strip()
# If a prediction exists, any prior API error is stale.
results_df.loc[results_df["prediction"] != "", "error"] = ""
missing_count = int(results_df["prediction"].eq("").sum())
error_count = int(results_df["error"].ne("").sum())
print("Total rows:", len(results_df))
print("Predictions present:", int(results_df["prediction"].ne("").sum()))
print("Missing predictions:", missing_count)
print("Unresolved errors:", error_count)
if not TEST_MODE and (len(results_df) != 250 or missing_count or error_count):
    print("⚠️ Not final yet. Repair missing/failed rows before reporting.")
elif not missing_count and not error_count:
    print("✅ No missing predictions or unresolved API errors.")

In [ ]:

valid = results_df[results_df["ground_truth"].fillna("").astype(str).str.strip().ne("") & results_df["prediction"].fillna("").astype(str).str.strip().ne("")].copy()
if len(valid) == 0:
    print("No valid prediction/ground-truth pairs to evaluate.")
else:
    print(f"Overall exact-match accuracy: {valid['correct'].mean() * 100:.2f}%")
    variant_summary = valid.groupby(["dataset", "variant"])["correct"].agg(total="count", correct="sum", accuracy="mean").reset_index()
    variant_summary["accuracy_percent"] = variant_summary["accuracy"] * 100
    variant_summary = variant_summary[["dataset", "variant", "total", "correct", "accuracy_percent"]]
    display(variant_summary)

In [ ]:

if TEST_MODE:
    results_file = "experiment_04_chain_of_thought_Qwen_TEST_results.csv"
    summary_file = "experiment_04_chain_of_thought_Qwen_TEST_summary.csv"
else:
    results_file = "experiment_04_chain_of_thought_Qwen_250_results.csv"
    summary_file = "experiment_04_chain_of_thought_Qwen_250_summary.csv"
results_df.to_csv(results_file, index=False)
if len(valid) > 0:
    variant_summary.to_csv(summary_file, index=False)
print("Saved results:", results_file)
if len(valid) > 0:
    print("Saved summary:", summary_file)
if checkpoint_file:
    print("Checkpoint:", checkpoint_file)

In [ ]:

from google.colab import files
files.download(results_file)
time.sleep(2)
if len(valid) > 0:
    files.download(summary_file)
print("Download started.")